# 第 16 週 實作｜平衡點、穩定性與梯度流

整門課的匯合點。前半問「系統會停在哪」,後半揭穿:<strong>梯度下降就是在解微分方程,learning rate 就是步長</strong>——而「lr 太大會發散」和「Euler 步長太大會不穩定」是同一個數學。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜相線與穩定性:符號分析 vs 判準

觀念 3、4 的驗證。這格對幾個方程自動找平衡點、算 $f'$、判定穩定性,並用數值積分確認預測正確。


In [ ]:
from scipy.integrate import solve_ivp
y = sp.Symbol('y', real=True)

cases = [
    ("y(1-y)",            y*(1-y)),
    ("(y-2)(y+1)",        (y-2)*(y+1)),
    ("y(1-y)(2-y)",       y*(1-y)*(2-y)),
    ("-y**3",             -y**3),
]

for name, expr in cases:
    fp = sp.diff(expr, y)
    eqs = sorted([e for e in sp.solve(sp.Eq(expr, 0), y) if e.is_real])
    print(f"\nf(y) = {name}     f'(y) = {sp.simplify(fp)}")
    for e in eqs:
        d = sp.simplify(fp.subs(y, e))
        if d.is_number and d != 0:
            verdict = "穩定" if d < 0 else "不穩定"
            extra = f"  時間常數 τ={float(1/abs(d)):.3f}"
        else:
            # 判準失效 → 回去看兩側符號
            lo = float(expr.subs(y, e - sp.Rational(1,10)))
            hi = float(expr.subs(y, e + sp.Rational(1,10)))
            verdict = ("穩定(判準失效,由符號判定)" if lo > 0 and hi < 0
                       else "不穩定(判準失效,由符號判定)" if lo < 0 and hi > 0
                       else "半穩定(判準失效)")
            extra = f"  兩側 f: {lo:+.4f} / {hi:+.4f}"
        print(f"   y*={e}:  f'={d}  → {verdict}{extra}")

# 數值驗證:y' = y(1-y)(2-y) 的三個起點命運
f = lambda t, Y: Y[0]*(1-Y[0])*(2-Y[0])
print("\n數值驗證 y' = y(1-y)(2-y):")
for y0 in [0.5, 1.5, 2.5]:
    s = solve_ivp(f, [0, 8], [y0], rtol=1e-9, atol=1e-12)
    end = s.y[0, -1]
    fate = "→ 1" if abs(end-1) < 1e-3 else ("→ 發散" if end > 10 else f"→ {end:.4f}")
    print(f"  y0={y0}:  t=8 時 y={end:12.4f}   {fate}")

ts = np.linspace(0, 8, 200)
for y0, c in [(0.05, 'C0'), (0.5, 'C1'), (1.5, 'C2'), (2.2, 'C3')]:
    s = solve_ivp(f, [0, 8], [y0], dense_output=True, rtol=1e-9, atol=1e-12)
    vals = s.sol(ts)[0]
    plt.plot(ts, np.clip(vals, -0.5, 4), color=c, label=f'y0={y0}')
for e, style in [(0, ':'), (1, '--'), (2, ':')]:
    plt.axhline(e, color='k', ls=style, lw=1)
plt.ylim(-0.3, 3.5); plt.xlabel('t'); plt.ylabel('y'); plt.legend(fontsize=8)
plt.title("y=1 attracts (0,2); y=0 and y=2 repel")
plt.show()

In [ ]:
# TODO 學生練習:加入 f(y) = y**2(半穩定的例子)
# 判準會失效。兩側的符號是什麼?從左邊和從右邊出發的命運一樣嗎?

## Lab 2｜梯度下降就是 Euler:數值上完全一致

觀念 6 的核心驗證。這格分別寫「Euler 解梯度流」與「梯度下降」兩支程式,確認它們產生<strong>逐位元相同</strong>的軌跡。


In [ ]:
def euler(f, y0, t0, t1, h):
    """W14 Lab2 的同一支 Euler"""
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h*f(t, y)); ts.append(t + h)
    return np.array(ts), np.array(ys)

def gradient_descent(dL, theta0, eta, steps):
    """標準梯度下降"""
    th = [theta0]
    for _ in range(steps):
        th.append(th[-1] - eta*dL(th[-1]))
    return np.array(th)

# L(theta) = A/2 * theta^2  →  L'(theta) = A*theta
A = 4.0
L  = lambda th: A/2 * th**2
dL = lambda th: A * th

eta, steps = 0.2, 12
_, euler_path = euler(lambda t, th: -dL(th), 1.0, 0, eta*steps, eta)
gd_path = gradient_descent(dL, 1.0, eta, steps)

print(f"L = {A}/2 * theta^2,  eta = h = {eta}\n")
print(f"{'n':>3} {'Euler 解梯度流':>18} {'梯度下降':>18} {'逐位元相同?':>12}")
for n in range(len(gd_path)):
    same = euler_path[n] == gd_path[n]
    print(f"{n:3d} {euler_path[n]:18.15f} {gd_path[n]:18.15f} {str(same):>12}")
print(f"\n全部逐位元相同? {np.array_equal(euler_path, gd_path)}")

# --- 穩定門檻 eta < 2/A ---
print(f"\n穩定門檻:eta < 2/A = {2/A}")
print(f"{'eta':>6} {'|1-eta*A|':>11} {'40 步後 |theta|':>16} {'狀態'}")
for e in [0.1, 0.25, 0.4, 0.49, 0.51, 0.6]:
    p = gradient_descent(dL, 1.0, e, 40)
    ratio = abs(1 - e*A)
    print(f"{e:6.2f} {ratio:11.3f} {abs(p[-1]):16.4e}   "
          f"{'收斂' if ratio < 1 else '發散'}"
          f"{'  (單調)' if 0 < 1-e*A else ('  (震盪)' if ratio < 1 else '')}")

# 連續軌跡 vs 離散步伐
tt = np.linspace(0, 2.4, 300)
plt.plot(tt, np.exp(-A*tt), 'k-', lw=2, label='gradient flow (exact)')
for e, c in [(0.1, 'C0'), (0.25, 'C1'), (0.45, 'C3')]:
    p = gradient_descent(dL, 1.0, e, int(2.4/e))
    plt.plot(np.arange(len(p))*e, p, 'o--', ms=4, color=c, label=f'GD eta={e}')
plt.xlabel('t  (= n * eta)'); plt.ylabel('theta'); plt.legend(fontsize=8)
plt.title('Gradient descent tracks the gradient flow')
plt.show()

In [ ]:
# TODO 學生練習:把 A 改成 20。穩定門檻變成多少?
# 再試 eta = 1/A(=0.05)。會發生什麼?(提示:公比 1-eta*A = ?)

## Lab 3｜momentum 的三種阻尼

觀念 8 說 momentum 是帶阻尼的二階 ODE,臨界阻尼 $\gamma=2\sqrt A$ 最快。這格把過阻尼、臨界、欠阻尼三種情形一起跑出來。


In [ ]:
from scipy.integrate import solve_ivp

A = 4.0
gamma_crit = 2*math.sqrt(A)
print(f"A = {A},  臨界阻尼 gamma = 2*sqrt(A) = {gamma_crit}\n")

# theta'' + gamma*theta' + A*theta = 0  →  一階系統 [theta, v]
def damped(gamma):
    return lambda t, Y: [Y[1], -gamma*Y[1] - A*Y[0]]

ts = np.linspace(0, 6, 500)
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
print(f"{'gamma':>7} {'判別式 g^2-4A':>14} {'類型':>10} {'t=6 時 |theta|':>16}")
for gamma, label, c in [(6.0, 'over-damped', 'C0'),
                        (gamma_crit, 'critical', 'C2'),
                        (1.0, 'under-damped', 'C3')]:
    s = solve_ivp(damped(gamma), [0, 6], [1.0, 0.0], dense_output=True,
                  rtol=1e-10, atol=1e-12)
    plt.plot(ts, s.sol(ts)[0], color=c, lw=1.8, label=f'{label} (g={gamma:.1f})')
    disc = gamma**2 - 4*A
    print(f"{gamma:7.2f} {disc:14.2f} {label:>14} {abs(s.sol(6)[0]):16.3e}")
plt.axhline(0, color='k', lw=0.6); plt.legend(fontsize=8)
plt.xlabel('t'); plt.ylabel('theta'); plt.title('Three damping regimes')

# --- 離散版:momentum vs 純梯度下降,在一個「長谷」上 ---
# L(x,y) = (x^2 + 20*y^2)/2  —— 條件數 20 的 ill-conditioned 谷
Ax, Ay = 1.0, 20.0
dL = lambda p: np.array([Ax*p[0], Ay*p[1]])
eta = 2/Ay * 0.9                        # 由最陡方向決定的上限

def run(beta, steps=120):
    p, v, path = np.array([10.0, 1.0]), np.zeros(2), []
    for _ in range(steps):
        v = beta*v - eta*dL(p)
        p = p + v
        path.append(p.copy())
    return np.array(path)

plt.subplot(1, 2, 2)
for beta, c in [(0.0, 'C0'), (0.9, 'C3')]:
    path = run(beta)
    lab = 'plain GD' if beta == 0 else f'momentum beta={beta}'
    plt.semilogy(np.abs(path[:, 0]), color=c, label=lab)
plt.xlabel('step'); plt.ylabel('|x| (the flat direction)')
plt.legend(fontsize=8); plt.title(f'Flat direction: eta capped by the steep one')
plt.tight_layout(); plt.show()

for beta in [0.0, 0.9]:
    path = run(beta)
    print(f"beta={beta}: 120 步後 |x| = {abs(path[-1,0]):.4e}"
          f"   {'(純梯度下降,平坦方向幾乎不動)' if beta==0 else '(momentum 累積速度,快得多)'}")

In [ ]:
# TODO 學生練習:把 Ay 改成 100(條件數 100)
# 純梯度下降需要多少步才能讓 |x| < 0.1?momentum 呢?
# 再試 beta = 0.99,會發生什麼?(提示:阻尼太小 → 欠阻尼震盪)